In [1]:
import pandas as pd
import os
import pydicom

# Read CSV
df = pd.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_label_coordinates.csv")
os.mkdir('images') 
# Output list
yolo_rows = []

# Box size in pixels (recommended for medical defect detection)
box_size = 5

for idx, row in df.iterrows():
    study_id = str(row["study_id"])
    series_id = str(row["series_id"])
    instance_number = str(row["instance_number"])
    condition = row["condition"]
    level = row["level"]
    x = float(row["x"])
    y = float(row["y"])

    # Build DICOM path
    dcm_path = os.path.join("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images",study_id, series_id, f"{instance_number}.dcm")
    
    # Read DICOM
    ds = pydicom.dcmread(dcm_path)
    image = ds.pixel_array
    height, width = image.shape
    ds.save_as(f'images/{study_id}_{series_id}_{instance_number}.jpg')
    # Normalize for YOLO format
    x_min = x-5
    y_min = y-5
    x_max = 5+x
    y_max = 5+y

    yolo_rows.append({
        "study_id": study_id,
        "series_id": series_id,
        "instance_number": instance_number,
        "condition": condition,
        "level": level,
        "x_min": x_min,
        "y_min": y_min,
        "x_max": x_max,
        "y_max": y_max,
        "class_id": 0,
        'img_width' : width,
        'img_height' : height# Can be mapped from `condition` later
    })

# Save to CSV
yolo_df = pd.DataFrame(yolo_rows)
yolo_df.to_csv("yolo_annotations.csv", index=False)
